# ARC-v0.33 — NQ-GTE One-Shot Decision-Regret Audit

**Scientific role:** post-primary, reviewer-oriented **decision-consequence audit** derived from the completed ARC-v0.32 exhaustive-reference experiment.

This notebook asks a practical evaluation question:

> **When one-shot retrieval quality cannot distinguish two approximation mechanisms, do their feedback-time outcomes remain interchangeable?**

The audit uses the already-frozen ARC-v0.32 NQ-GTE main subset and its exact-reference round-0 / H=4 outputs.

## Important evidence-status guardrail

This is **not a pristine prospective confirmation**. ARC-v0.32 source outcomes already exist, and this v0.33 analysis is being specified after those source artifacts were produced. The notebook freezes the *derived analysis formulas* and retains all outcomes, but the result must be reported as **post-primary / reviewer-oriented analysis**.

No new retrieval, re-encoding, ANN tuning, query selection, feedback-policy tuning, or outcome-driven threshold search is performed.

## Primary estimand

For query \(q\), define one-shot exhaustive-reference absolute utility gaps

\[
g^{(0)}_{\mathrm{rep}}(q)
=
|u_{\mathrm{Flat}}^{(0)} - u_{\mathrm{PQ32@64}}^{(0)}|
\]

and

\[
g^{(0)}_{\mathrm{search}}(q)
=
|u_{\mathrm{Flat}}^{(0)} - u_{\mathrm{SQ8@2}}^{(0)}|.
\]

The **exact one-shot ambiguity subset** is

\[
\mathcal A_0 =
\{q:\ |g^{(0)}_{\mathrm{rep}}(q)-g^{(0)}_{\mathrm{search}}(q)|\le 10^{-12}\}.
\]

Using ARC-v0.32's policy-averaged H=4 terminal exhaustive-reference gaps,

\[
g^{(H)}_{\mathrm{rep}}(q),\qquad
g^{(H)}_{\mathrm{search}}(q),
\]

the primary estimand is

\[
\Delta_{\mathrm{amb}}
=
\mathbb E_{q\in\mathcal A_0}
[
g^{(H)}_{\mathrm{rep}}(q)
-
g^{(H)}_{\mathrm{search}}(q)
].
\]

Inference: 10,000-replicate query bootstrap.

Interpretation:
- CI \(>0\): among queries that one-shot nDCG cannot distinguish, representation approximation has a larger terminal absolute gap on average.
- CI \(<0\): the opposite ordering.
- CI crossing 0: unresolved.

This is an **evaluation consequence**, not a causal theorem and not a deployment-cost claim.

## Secondary analyses

1. Fraction of exact one-shot ties that become terminal-decisive.
2. Mean absolute terminal mechanism separation within the ambiguity subset.
3. One-shot selector mis-selection rate and regret on one-shot-decisive queries.
4. Sensitivity across one-shot equivalence tolerances \(\tau \in \{0, .01, .025, .05, .10\}\).
5. Sensitivity across exact-reference and the original relative-high comparator.
6. Deterministic artifact hashes and conservative manuscript wording.


In [ ]:
# Cell 1 — Imports and deterministic analysis settings
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os
import numpy as np
import pandas as pd

SEED = 20260833
BOOTSTRAP_REPS = 10_000
EPS_EXACT_TIE = 1e-12
TOLERANCE_GRID = [0.0, 0.01, 0.025, 0.05, 0.10]
TERMINAL_DECISION_EPS = 1e-12

EXPECTED_V032_PROTOCOL_SHA256 = (
    "54d876a30dd34ea258eba4414fa24b67"
    "d880487ccd446f2997d2979dd8d06397"
)
EXPECTED_MAIN_QUERIES = 500

rng = np.random.default_rng(SEED)

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def bootstrap_mean(x, reps=BOOTSTRAP_REPS, seed=SEED):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return {"n": 0, "mean": None, "ci95": [None, None]}
    rr = np.random.default_rng(seed)
    vals = np.empty(reps, dtype=np.float64)
    n = len(x)
    for i in range(reps):
        vals[i] = x[rr.integers(0, n, size=n)].mean()
    return {
        "n": int(n),
        "mean": float(x.mean()),
        "ci95": [float(v) for v in np.quantile(vals, [0.025, 0.975])],
    }

def bootstrap_fraction(x, reps=BOOTSTRAP_REPS, seed=SEED):
    return bootstrap_mean(np.asarray(x, dtype=float), reps=reps, seed=seed)

print("settings ready")


In [ ]:
# Cell 2 — Mount Drive and locate the completed ARC-v0.32 source run
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
except Exception:
    # Local fallback for manual testing: set V032_SOURCE_DIR environment variable.
    DRIVE_ROOT = None

V032_RUN_ID = "20260827-132934"

if DRIVE_ROOT is not None:
    ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"
    V032_SOURCE = (
        ARC_ROOT
        / "nq-gte-exhaustive-reference-v032"
        / V032_RUN_ID
    )
    V033_ROOT = ARC_ROOT / "nq-gte-one-shot-decision-regret-v033"
else:
    override = os.environ.get("V032_SOURCE_DIR")
    assert override, "Set V032_SOURCE_DIR when running outside Colab."
    V032_SOURCE = Path(override)
    V033_ROOT = V032_SOURCE.parent / "v033-local-output"

assert V032_SOURCE.is_dir(), V032_SOURCE

ROUND0_PATH = V032_SOURCE / "v032_round0_full_validation_exact_reference.csv"
QEND_PATH = V032_SOURCE / "v032_query_level_endpoints.csv"
V032_PROTOCOL_PATH = V032_SOURCE / "V032_FROZEN_PROTOCOL.json"
V032_PROTOCOL_SHA_PATH = V032_SOURCE / "V032_PROTOCOL_SHA256.txt"
V032_REPORT_PATH = V032_SOURCE / "v032_final_report.json"

for p in [
    ROUND0_PATH, QEND_PATH, V032_PROTOCOL_PATH,
    V032_PROTOCOL_SHA_PATH, V032_REPORT_PATH
]:
    assert p.is_file(), p

v032_sha_text = V032_PROTOCOL_SHA_PATH.read_text(encoding="utf-8").strip()
assert EXPECTED_V032_PROTOCOL_SHA256 in v032_sha_text, v032_sha_text
assert sha256_file(V032_PROTOCOL_PATH) == EXPECTED_V032_PROTOCOL_SHA256

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = V033_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

print("V032_SOURCE:", V032_SOURCE)
print("OUT:", OUT)


In [ ]:
# Cell 3 — Freeze the v0.33 derived-analysis protocol
protocol = {
    "study_id": "ARC-v0.33",
    "title": "NQ-GTE One-Shot Decision-Regret Audit",
    "scientific_role": "post-primary reviewer-oriented derived analysis",
    "prospective_status": (
        "NOT_OUTCOME_BLIND: source ARC-v0.32 outcomes already existed before "
        "this derived audit was specified"
    ),
    "source_study": "ARC-v0.32",
    "source_run_id": V032_RUN_ID,
    "source_protocol_sha256": EXPECTED_V032_PROTOCOL_SHA256,
    "dataset": "BEIR Natural Questions",
    "encoder": "thenlper/gte-small",
    "horizon": 4,
    "operator": "anchored",
    "main_queries_expected": EXPECTED_MAIN_QUERIES,
    "policy_aggregation": (
        "ARC-v0.32 query-level endpoint table; terminal gaps averaged over "
        "the frozen 8-policy structural subset"
    ),
    "reference_primary": (
        "exhaustive FlatIP over persisted shared normalized corpus-embedding store"
    ),
    "representation_low": "IVF-PQ32 @ nprobe=64",
    "search_effort_low": "IVF-SQ8 @ nprobe=2",
    "relative_high": "IVF-SQ8 @ nprobe=64",
    "primary_subset": (
        "queries with |round0_exact_gap_rep - round0_exact_gap_search| <= 1e-12"
    ),
    "primary_estimand": (
        "mean(rep_exact_terminal_gap - search_exact_terminal_gap) "
        "within the exact one-shot ambiguity subset"
    ),
    "primary_inference": "10000-replicate query bootstrap, two-sided 95% CI",
    "secondary": [
        "fraction of exact one-shot ties that become terminal-decisive",
        "mean absolute terminal mechanism separation among exact one-shot ties",
        "one-shot selector mis-selection rate on queries decisive at round0 and terminal",
        "one-shot selector terminal regret on round0-decisive queries",
        "one-shot-equivalence tolerance sweep tau={0,.01,.025,.05,.10}",
        "relative-high comparator sensitivity using SQ8@64 as reference",
    ],
    "tie_epsilon": EPS_EXACT_TIE,
    "terminal_decision_epsilon": TERMINAL_DECISION_EPS,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "seed": SEED,
    "retention_rule": (
        "Retain positive, null, reversed, or small effects. "
        "Do not redefine subsets or thresholds after derived outcomes are computed."
    ),
    "claim_guardrail": (
        "This audit quantifies offline evaluation consequences. It does not establish "
        "production routing value, equal system cost, answer-generation quality, "
        "or a universal causal mechanism."
    ),
}

PROTOCOL_PATH = OUT / "V033_ANALYSIS_PROTOCOL.json"
PROTOCOL_PATH.write_text(
    json.dumps(protocol, indent=2, sort_keys=True),
    encoding="utf-8",
)
protocol_sha = sha256_file(PROTOCOL_PATH)
(OUT / "V033_PROTOCOL_SHA256.txt").write_text(
    f"{protocol_sha}  {PROTOCOL_PATH.name}\n",
    encoding="utf-8",
)

print("V033 protocol SHA-256:", protocol_sha)


In [ ]:
# Cell 4 — Load and validate the two completed ARC-v0.32 source tables
r0 = pd.read_csv(ROUND0_PATH)
qe = pd.read_csv(QEND_PATH)

expected_r0 = {
    "query_id",
    "ndcg_exact",
    "ndcg_sq8_np64",
    "ndcg_sq8_np2",
    "ndcg_pq32_np64",
    "sq8_np64_abs_ndcg_gap_to_exact",
    "sq8_np2_abs_ndcg_gap_to_exact",
    "pq32_np64_abs_ndcg_gap_to_exact",
}
expected_qe = {
    "query_id",
    "rep_rel_terminal_gap",
    "search_rel_terminal_gap",
    "rep_exact_terminal_gap",
    "search_exact_terminal_gap",
}

assert expected_r0.issubset(r0.columns), sorted(expected_r0 - set(r0.columns))
assert expected_qe.issubset(qe.columns), sorted(expected_qe - set(qe.columns))
assert qe["query_id"].is_unique
assert r0["query_id"].is_unique
assert len(qe) == EXPECTED_MAIN_QUERIES, len(qe)

d = qe.merge(r0, on="query_id", how="inner", validate="one_to_one")
assert len(d) == EXPECTED_MAIN_QUERIES, len(d)

# Exact-reference round-0 gaps.
d["g0_rep_exact"] = d["pq32_np64_abs_ndcg_gap_to_exact"].astype(float)
d["g0_search_exact"] = d["sq8_np2_abs_ndcg_gap_to_exact"].astype(float)

# Relative-high round-0 gaps; aligned with the original SQ8@64 comparator.
d["g0_rep_rel"] = np.abs(
    d["ndcg_sq8_np64"].astype(float) - d["ndcg_pq32_np64"].astype(float)
)
d["g0_search_rel"] = np.abs(
    d["ndcg_sq8_np64"].astype(float) - d["ndcg_sq8_np2"].astype(float)
)

# H=4 terminal gaps, already policy-averaged by ARC-v0.32.
d["gH_rep_exact"] = d["rep_exact_terminal_gap"].astype(float)
d["gH_search_exact"] = d["search_exact_terminal_gap"].astype(float)
d["gH_rep_rel"] = d["rep_rel_terminal_gap"].astype(float)
d["gH_search_rel"] = d["search_rel_terminal_gap"].astype(float)

for c in [
    "g0_rep_exact","g0_search_exact","g0_rep_rel","g0_search_rel",
    "gH_rep_exact","gH_search_exact","gH_rep_rel","gH_search_rel",
]:
    assert np.isfinite(d[c]).all(), c

print("main queries:", len(d))
print(
    d[
        ["g0_rep_exact","g0_search_exact","gH_rep_exact","gH_search_exact"]
    ].describe().loc[["mean","50%"]]
)


In [ ]:
# Cell 5 — Primary exact-reference one-shot ambiguity analysis
d["exact_round0_diff"] = d["g0_rep_exact"] - d["g0_search_exact"]
d["exact_terminal_diff"] = d["gH_rep_exact"] - d["gH_search_exact"]

amb = d[np.abs(d["exact_round0_diff"]) <= EPS_EXACT_TIE].copy()
assert len(amb) > 0

amb["terminal_decisive"] = (
    np.abs(amb["exact_terminal_diff"]) > TERMINAL_DECISION_EPS
)
amb["terminal_abs_separation"] = np.abs(amb["exact_terminal_diff"])

primary_boot = bootstrap_mean(
    amb["exact_terminal_diff"].to_numpy(float),
    seed=SEED + 1,
)
abs_sep_boot = bootstrap_mean(
    amb["terminal_abs_separation"].to_numpy(float),
    seed=SEED + 2,
)
terminal_decisive_boot = bootstrap_fraction(
    amb["terminal_decisive"].astype(float).to_numpy(),
    seed=SEED + 3,
)

lo, hi = primary_boot["ci95"]
if lo > 0:
    primary_classification = "SEARCH_EFFORT_LOWER_TERMINAL_GAP_ON_ONESHOT_TIES"
elif hi < 0:
    primary_classification = "REPRESENTATION_LOWER_TERMINAL_GAP_ON_ONESHOT_TIES"
else:
    primary_classification = "UNRESOLVED"

primary_gate = {
    "study_id": "ARC-v0.33",
    "evidence_status": "POST_PRIMARY_DERIVED_ANALYSIS",
    "n_main_queries": int(len(d)),
    "n_exact_one_shot_ties": int(len(amb)),
    "exact_one_shot_tie_fraction": float(len(amb) / len(d)),
    "primary": {
        "estimand": (
            "mean(rep_exact_terminal_gap - search_exact_terminal_gap) "
            "among exact one-shot ties"
        ),
        **primary_boot,
        "classification": primary_classification,
    },
    "secondary_exact_tie_consequence": {
        "terminal_decisive_fraction": terminal_decisive_boot,
        "mean_absolute_terminal_mechanism_separation": abs_sep_boot,
    },
    "source_protocol_sha256": EXPECTED_V032_PROTOCOL_SHA256,
    "v033_protocol_sha256": protocol_sha,
}

(OUT / "v033_primary_gate.json").write_text(
    json.dumps(primary_gate, indent=2),
    encoding="utf-8",
)

print(json.dumps(primary_gate, indent=2))


In [ ]:
# Cell 6 — One-shot selector mis-selection and regret
def add_selector_columns(frame, ref):
    x = frame.copy()
    g0r, g0s = f"g0_rep_{ref}", f"g0_search_{ref}"
    ghr, ghs = f"gH_rep_{ref}", f"gH_search_{ref}"

    x[f"{ref}_round0_decisive"] = np.abs(x[g0r] - x[g0s]) > EPS_EXACT_TIE
    x[f"{ref}_terminal_decisive"] = np.abs(x[ghr] - x[ghs]) > TERMINAL_DECISION_EPS

    x[f"{ref}_oneshot_choice"] = np.where(
        x[g0r] < x[g0s], "representation",
        np.where(x[g0s] < x[g0r], "search_effort", "tie")
    )
    x[f"{ref}_terminal_oracle_choice"] = np.where(
        x[ghr] < x[ghs], "representation",
        np.where(x[ghs] < x[ghr], "search_effort", "tie")
    )

    x[f"{ref}_selected_terminal_gap"] = np.where(
        x[f"{ref}_oneshot_choice"] == "representation", x[ghr],
        np.where(
            x[f"{ref}_oneshot_choice"] == "search_effort", x[ghs], np.nan
        )
    )
    x[f"{ref}_oracle_terminal_gap"] = np.minimum(x[ghr], x[ghs])
    x[f"{ref}_selector_regret"] = (
        x[f"{ref}_selected_terminal_gap"] - x[f"{ref}_oracle_terminal_gap"]
    )
    return x

d = add_selector_columns(d, "exact")
d = add_selector_columns(d, "rel")

selector_rows = []
for j, ref in enumerate(["exact", "rel"]):
    decisive = d[d[f"{ref}_round0_decisive"]].copy()
    both = decisive[decisive[f"{ref}_terminal_decisive"]].copy()

    mis = (
        both[f"{ref}_oneshot_choice"]
        != both[f"{ref}_terminal_oracle_choice"]
    ).astype(float)

    selector_rows.append({
        "reference": ref,
        "n_total": int(len(d)),
        "n_round0_decisive": int(len(decisive)),
        "round0_decisive_fraction": float(len(decisive) / len(d)),
        "n_round0_and_terminal_decisive": int(len(both)),
        "misselection_rate": float(mis.mean()) if len(mis) else np.nan,
        "misselection_ci_lo": (
            bootstrap_fraction(mis.to_numpy(), seed=SEED + 10 + j)["ci95"][0]
            if len(mis) else np.nan
        ),
        "misselection_ci_hi": (
            bootstrap_fraction(mis.to_numpy(), seed=SEED + 20 + j)["ci95"][1]
            if len(mis) else np.nan
        ),
        "mean_selector_regret": float(
            decisive[f"{ref}_selector_regret"].mean()
        ),
        "selector_regret_ci_lo": bootstrap_mean(
            decisive[f"{ref}_selector_regret"].to_numpy(),
            seed=SEED + 30 + j,
        )["ci95"][0],
        "selector_regret_ci_hi": bootstrap_mean(
            decisive[f"{ref}_selector_regret"].to_numpy(),
            seed=SEED + 40 + j,
        )["ci95"][1],
        "mean_selected_terminal_gap": float(
            decisive[f"{ref}_selected_terminal_gap"].mean()
        ),
        "mean_oracle_terminal_gap": float(
            decisive[f"{ref}_oracle_terminal_gap"].mean()
        ),
        "always_rep_terminal_gap": float(
            decisive[f"gH_rep_{ref}"].mean()
        ),
        "always_search_terminal_gap": float(
            decisive[f"gH_search_{ref}"].mean()
        ),
    })

selector_summary = pd.DataFrame(selector_rows)
selector_summary.to_csv(OUT / "v033_selector_regret_summary.csv", index=False)
selector_summary


In [ ]:
# Cell 7 — Tolerance sweep: when one-shot says "approximately equivalent"
tol_rows = []

for ref in ["exact", "rel"]:
    round0_diff = d[f"g0_rep_{ref}"] - d[f"g0_search_{ref}"]
    terminal_diff = d[f"gH_rep_{ref}"] - d[f"gH_search_{ref}"]

    for tau in TOLERANCE_GRID:
        x = d[np.abs(round0_diff) <= tau].copy()
        td = (
            x[f"gH_rep_{ref}"].to_numpy(float)
            - x[f"gH_search_{ref}"].to_numpy(float)
        )
        decisive = np.abs(td) > TERMINAL_DECISION_EPS

        signed = bootstrap_mean(
            td,
            seed=SEED + 100 + int(round(tau * 1000))
            + (0 if ref == "exact" else 1000),
        )
        absolute = bootstrap_mean(
            np.abs(td),
            seed=SEED + 200 + int(round(tau * 1000))
            + (0 if ref == "exact" else 1000),
        )
        dec = bootstrap_fraction(
            decisive.astype(float),
            seed=SEED + 300 + int(round(tau * 1000))
            + (0 if ref == "exact" else 1000),
        )

        tol_rows.append({
            "reference": ref,
            "tau": tau,
            "n_equivalent": int(len(x)),
            "equivalent_fraction": float(len(x) / len(d)),
            "terminal_decisive_fraction": float(dec["mean"]),
            "terminal_decisive_ci_lo": dec["ci95"][0],
            "terminal_decisive_ci_hi": dec["ci95"][1],
            "mean_signed_terminal_diff_rep_minus_search": signed["mean"],
            "signed_ci_lo": signed["ci95"][0],
            "signed_ci_hi": signed["ci95"][1],
            "mean_abs_terminal_mechanism_separation": absolute["mean"],
            "abs_sep_ci_lo": absolute["ci95"][0],
            "abs_sep_ci_hi": absolute["ci95"][1],
        })

tol_df = pd.DataFrame(tol_rows)
tol_df.to_csv(OUT / "v033_tolerance_sweep.csv", index=False)
tol_df


In [ ]:
# Cell 8 — Terminal-separation threshold audit within exact one-shot ties
thresholds = [0.0, 0.01, 0.025, 0.05, 0.10]
sep_rows = []

abs_td = np.abs(amb["exact_terminal_diff"].to_numpy(float))
for delta in thresholds:
    event = abs_td > delta
    b = bootstrap_fraction(
        event.astype(float),
        seed=SEED + 500 + int(round(delta * 1000)),
    )
    sep_rows.append({
        "terminal_separation_threshold": delta,
        "fraction_exceeding": b["mean"],
        "ci_lo": b["ci95"][0],
        "ci_hi": b["ci95"][1],
        "n_exact_one_shot_ties": int(len(amb)),
    })

sep_df = pd.DataFrame(sep_rows)
sep_df.to_csv(
    OUT / "v033_exact_tie_terminal_separation_thresholds.csv",
    index=False,
)
sep_df


In [ ]:
# Cell 9 — Save query-level decision table
keep = [
    "query_id",
    "g0_rep_exact","g0_search_exact","gH_rep_exact","gH_search_exact",
    "exact_round0_diff","exact_terminal_diff",
    "exact_round0_decisive","exact_terminal_decisive",
    "exact_oneshot_choice","exact_terminal_oracle_choice",
    "exact_selected_terminal_gap","exact_oracle_terminal_gap",
    "exact_selector_regret",
    "g0_rep_rel","g0_search_rel","gH_rep_rel","gH_search_rel",
    "rel_round0_decisive","rel_terminal_decisive",
    "rel_oneshot_choice","rel_terminal_oracle_choice",
    "rel_selected_terminal_gap","rel_oracle_terminal_gap",
    "rel_selector_regret",
]
d[keep].sort_values("query_id").to_csv(
    OUT / "v033_query_decision_table.csv",
    index=False,
)
print("saved query-level decision table:", len(d))


In [ ]:
# Cell 10 — Optional reviewer-facing figures
import matplotlib.pyplot as plt

# Figure A: one-shot mechanism difference vs terminal mechanism difference.
plt.figure(figsize=(6.4, 4.8))
plt.scatter(
    d["exact_round0_diff"],
    d["exact_terminal_diff"],
    s=16,
    alpha=0.6,
)
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel("Round-0 exact-reference gap difference (representation − search)")
plt.ylabel("H=4 terminal gap difference (representation − search)")
plt.title("One-shot preference does not fully determine feedback-time preference")
plt.tight_layout()
plt.savefig(OUT / "v033_round0_vs_terminal_mechanism_difference.png", dpi=220)
plt.show()

# Figure B: tolerance sweep for exact-reference equivalence subset.
x = tol_df[tol_df["reference"] == "exact"].copy()
plt.figure(figsize=(6.4, 4.8))
plt.plot(
    x["tau"],
    x["mean_abs_terminal_mechanism_separation"],
    marker="o",
)
plt.xlabel("One-shot equivalence tolerance τ")
plt.ylabel("Mean |terminal mechanism gap difference|")
plt.title("Terminal separation among one-shot-equivalent queries")
plt.tight_layout()
plt.savefig(OUT / "v033_tolerance_vs_terminal_separation.png", dpi=220)
plt.show()


In [ ]:
# Cell 11 — Final conservative report and manuscript-facing wording
primary = primary_gate["primary"]
tie_frac = primary_gate["exact_one_shot_tie_fraction"]
terminal_decisive = primary_gate[
    "secondary_exact_tie_consequence"
]["terminal_decisive_fraction"]["mean"]
abs_sep = primary_gate[
    "secondary_exact_tie_consequence"
]["mean_absolute_terminal_mechanism_separation"]["mean"]

exact_sel = selector_summary[
    selector_summary["reference"] == "exact"
].iloc[0].to_dict()

if primary["classification"] == "SEARCH_EFFORT_LOWER_TERMINAL_GAP_ON_ONESHOT_TIES":
    wording = (
        "In a post-primary NQ-GTE decision-consequence audit, "
        f"{tie_frac:.1%} of the frozen 500-query sample had identical round-0 "
        "nDCG@10 loss under the two low-fidelity mechanisms relative to the "
        "exhaustive reference. Despite this one-shot ambiguity, "
        f"{terminal_decisive:.1%} of those queries became terminally distinguishable "
        "after H=4 anchored feedback. Within the one-shot-tied subset, the "
        "policy-averaged terminal absolute nDCG@10 gap was larger for representation "
        f"approximation by {primary['mean']:.4f} on average "
        f"(95% bootstrap CI [{primary['ci95'][0]:.4f}, {primary['ci95'][1]:.4f}]). "
        "This demonstrates a concrete offline evaluation consequence: equality under "
        "one-shot effectiveness does not imply equality under feedback-time behavior. "
        "The analysis is post-primary and does not establish production cost or "
        "answer-generation consequences."
    )
elif primary["classification"] == "REPRESENTATION_LOWER_TERMINAL_GAP_ON_ONESHOT_TIES":
    wording = (
        "In the post-primary NQ-GTE decision-consequence audit, exact one-shot ties "
        "do not remain behaviorally equivalent under H=4 feedback; the tied subset "
        "instead favors representation approximation at the terminal gap endpoint. "
        "This is retained as a directionally different boundary and is not treated "
        "as prospective confirmation."
    )
else:
    wording = (
        "In the post-primary NQ-GTE decision-consequence audit, exact one-shot ties "
        "show measurable terminal separation, but the signed representation-versus-"
        "search ordering remains statistically unresolved. This supports one-shot "
        "incompleteness without a directional terminal claim."
    )

final_report = {
    "study_id": "ARC-v0.33",
    "status": "COMPLETE",
    "evidence_status": "POST_PRIMARY_DERIVED_ANALYSIS",
    "source_study": "ARC-v0.32",
    "source_run_id": V032_RUN_ID,
    "n_main_queries": int(len(d)),
    "primary": primary_gate,
    "exact_reference_selector_summary": exact_sel,
    "suggested_manuscript_wording": wording,
    "claim_guardrail": protocol["claim_guardrail"],
    "source_protocol_sha256": EXPECTED_V032_PROTOCOL_SHA256,
    "v033_protocol_sha256": protocol_sha,
}

(OUT / "v033_final_report.json").write_text(
    json.dumps(final_report, indent=2, default=float),
    encoding="utf-8",
)

print(wording)


In [ ]:
# Cell 12 — Artifact integrity manifest
records = []
for p in sorted(OUT.iterdir()):
    if p.is_file() and p.name != "V033_ARTIFACT_SHA256.csv":
        records.append({
            "file": p.name,
            "bytes": int(p.stat().st_size),
            "sha256": sha256_file(p),
        })

manifest = pd.DataFrame(records)
manifest.to_csv(OUT / "V033_ARTIFACT_SHA256.csv", index=False)

print("ARC-v0.33 complete")
print("OUT:", OUT)
print(manifest[["file", "bytes"]].to_string(index=False))


## Reporting guardrails

Allowed claims should remain narrow:

- **Allowed:** one-shot equality/equivalence can leave meaningful feedback-time ambiguity in the audited NQ-GTE setting.
- **Allowed:** quantify terminal separation, selector regret, and mis-selection as an offline evaluation consequence.
- **Allowed:** report exact-reference and relative-high sensitivity separately.
- **Not allowed:** claim this is a new untouched confirmation.
- **Not allowed:** claim the two mechanisms have equal deployment cost.
- **Not allowed:** claim terminal retrieval-gap regret equals answer-quality regret.
- **Not allowed:** claim a universal theorem for ANN feedback systems.

If this result is incorporated into the SIGIR manuscript, label it **post-primary decision-consequence analysis**.
